# 🧠 Visual Cortex Brain Encoding Pipeline
### Which Deep-Net Layers Explain Which Human Visual Regions?
**Algonauts 2023 / Natural Scenes Dataset (NSD) Reproduction & Extension**

This notebook runs cleanly on a single **T4 GPU** on Google Colab or Kaggle.
- **Hierarchy**: Layer depth vs cortical stream rank
- **Training Objective**: Supervised (DeiT-S) vs Self-Supervised (DINO) vs Language-Image (CLIP)
- **Controls**: Untrained ResNet-50 (inductive bias) and Multiscale Gabor pyramid (pixels)
- **Selectivity**: Category tuning in FFA, PPA, and EBA
- **Robustness**: Cross-subject replication across Subjects 1–4

## 1. Setup & Environment Installation

In [ ]:
# Install dependencies in Colab/Kaggle
!pip install -q timm open-clip-torch scikit-learn seaborn pyyaml

import os

# Auto-detect Algonauts dataset path (Kaggle or local)
candidate_dirs = [
    '/kaggle/input/algonauts-2023-challenge',
    '/kaggle/input/algonauts-2023-tutorial-data',
    'data/raw/algonauts_2023'
]

data_dir = None
for d in candidate_dirs:
    if os.path.exists(os.path.join(d, 'subj01')):
        data_dir = d
        break

if data_dir is None:
    raise FileNotFoundError(
        'Algonauts 2023 dataset not found! Please attach the dataset in Kaggle Inputs or place in data/raw/algonauts_2023/.'
    )
print('Found real Algonauts dataset at:', data_dir)


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch version: {torch.__version__}, Running on: {device}')
if device == 'cuda':
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

import torch
from features.extract import extract_model_features

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Extracting on: {device}')

models_to_run = [
    'alexnet',
    'resnet50',
    'deit_small',
    'clip_vit_b16',
    'resnet50_untrained'
]

for m in models_to_run:
    print(f'=== Extracting features for: {m} ===')
    extract_model_features(model_key=m, subject_id='subj01', data_root=data_dir, batch_size=64, device=device)


In [ ]:
import os
from data.mock_data import generate_synthetic_dataset

# If using Kaggle with input attached:
# data_dir = '/kaggle/input/algonauts-2023-tutorial-data'

data_dir = 'data/raw/algonauts_2023'
if not os.path.exists(os.path.join(data_dir, 'subj01')):
    print('Generating functional quickstart dataset for demo...')
    generate_synthetic_dataset(output_dir=data_dir, subject_id='subj01', n_images=200, n_lh_vertices=1000, n_rh_vertices=1000)
else:
    print('Found Algonauts data in:', data_dir)

## 4. Feature Extraction & Caching
Extract 8 layer taps per model with spatial pooling and save as float16.

In [ ]:
import torch
from features.extract import extract_model_features

# Auto-detect device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Extracting on: {device}')

models_to_run = [
    'gabor_pyramid',
    'alexnet',
    'resnet50',
    'resnet50_untrained'
]

for m in models_to_run:
    print(f'=== Extracting features for: {m} ===')
    extract_model_features(model_key=m, subject_id='subj01', data_root=data_dir, batch_size=64, device=device)

## 5. Fit Closed-Form SVD Ridge Encoding Models
- 1024-dim Train-only PCA
- Inner 5-fold CV over log-alpha grid
- Out-of-sample evaluation on held-out test split
- Shuffled null distribution check

In [ ]:
from encoding.pipeline import EncodingPipeline

pipeline = EncodingPipeline(subject_id='subj01', data_root=data_dir)
results_summary = pipeline.run_full_subject(selected_models=models_to_run)

## 6. Analysis: Hierarchy & Model Comparison

In [ ]:
from analysis.hierarchy import compute_peak_layer_per_roi, evaluate_hierarchy_correlation

peaks = compute_peak_layer_per_roi(results_summary, 'resnet50')
h_eval = evaluate_hierarchy_correlation(peaks)
print(f'ResNet-50 Hierarchy Spearman rho: {h_eval["spearman_rho"]:.3f} (p = {h_eval["p_value"]:.4f})')
print(f'95% Bootstrap Confidence Interval: {h_eval["ci_95"]}')

## 7. Generate Static Figures

In [ ]:
from analysis.plotting import plot_layer_roi_heatmap, plot_model_comparison_per_roi

plot_layer_roi_heatmap(results_summary, 'resnet50', output_path='figures/fig1_layer_roi_heatmap.png')

best_scores = {
    'ResNet-50': {r: peaks[r]['max_r'] for r in peaks},
}
plot_model_comparison_per_roi(best_scores, output_path='figures/fig2_model_comparison.png')
print('Figures generated in figures/ directory!')